<a href="https://colab.research.google.com/github/Shea-Fyffe/transforming-personality-item-generation/blob/main/generate-dpo-items.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade transformers[sentencepiece]

In [ ]:
#@title Import Libraries
# standard
import os
from dataclasses import (dataclass, field, fields)
import json

# application specific
from transformers import (AutoTokenizer, pipeline)
from peft import (AutoPeftModelForCausalLM)

In [ ]:
#@title Prevent CUDA fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
#@title Model Class
@dataclass
class AIGobj:
    trained_model_dir: str = "/content/drive/MyDrive/Colab Notebooks/dissertation/study1/ouput-dpo"
    model_kwargs: dict[str, any] = field(default_factory=dict)
    tokenizer_kwargs: dict[str, any] = field(default_factory=dict)
    system_prompt: str | None = None
    _infer_model: AutoPeftModelForCausalLM = field(init=False)
    _infer_tokenizer: AutoTokenizer = field(init=False)
    _infer_pipe: pipeline = field(init=False)

    def _setup_pipeline(self):
        if not os.path.exists(self.trained_model_dir):
            raise FileNotFoundError(f"{self.trained_model_dir} must be valid path model directory.")
        # Load Model with PEFT adapter
        self._infer_model = AutoPeftModelForCausalLM.from_pretrained(self.trained_model_dir, **self.get_model_kwargs)
        self._infer_tokenizer = AutoTokenizer.from_pretrained(self.trained_model_dir)
        # load pipeline
        self._infer_pipe = pipeline("text-generation", model = self._infer_model ,
                                    tokenizer = self._infer_tokenizer)

    ## Post-Init
    def __post_init__(self):
        self._setup_pipeline()

    ## Inference Function
    def infer(self, prompts: list[str] | str, add_system_prompt: bool = True, inference_kwargs: dict | None = None, see_prompts: bool = False):
        if not hasattr(self, '_infer_pipe'):
            self._setup_pipeline()
        def _as_message(_prompt: str, system_prompt: bool) -> list[dict]:
            return [{"role": "system", "content": self.get_system_prompt}, {"role":"user", "content": _prompt}] if system_prompt else [{"role":"user", "content": _prompt}]
        _INFER_MSGS = []
        if isinstance(prompts, str):
            prompts = [prompts]
        for prompt in prompts:
            _INFER_MSGS.append(self._infer_pipe.tokenizer.apply_chat_template(_as_message(prompt, add_system_prompt), tokenize = False, add_generation_prompt = True))
        if see_prompts:
            return _INFER_MSGS
        if not inference_kwargs:
            inference_kwargs = self.get_infer_kwargs
        return [self._infer_pipe(_MSG, **inference_kwargs) for _MSG in _INFER_MSGS]

    ## Properties
    @property
    def get_model_kwargs(self) -> dict[str, any]:
        return self.default_model_kwargs | self.model_kwargs

    @property
    def get_system_prompt(self) -> dict[str, any]:
        return self.default_inference_sys_prompt if self.system_prompt is None else self.system_prompt

    @property
    def get_tokenizer_kwargs(self) -> dict[str, any]:
        return self.tokenizer_kwargs

    @property
    def get_infer_kwargs(self) -> dict[str, any]:
        return self.default_pipeline_infer_kwargs

    @property
    def default_model_kwargs(self) -> dict[str, any]:
        return {"device_map": "auto", "torch_dtype": torch.bfloat16,}

    @property
    def default_pipeline_infer_kwargs(self) -> dict[str, any]:
        return {"max_new_tokens": 2048, "temperature": .50, "eos_token_id": [self._infer_tokenizer.eos_token_id, self._infer_tokenizer.convert_tokens_to_ids("<|eot_id|>")], "pad_token_id": self._infer_tokenizer.pad_token_id}

    @property
    def default_inference_sys_prompt(self) -> str:
        return 'You are a test and survey developer who generates new survey items based on instructions, item specifications, constraints, examples, and a JSON response schema. Generate only items based on the JSON schema provided. Do *not* repeat instructions, examples, and constraints in the prompt.'

In [ ]:
def get_dpo_gen_prompts(json_file_path: str = "/content/drive/My Drive/Academic/research/dissertation/NLP-in-Personality/data/study1/dpo-inference-prompts--main.json") -> tuple[list[str], Dataset]:
    if not os.path.exists(json_file_path):
        from google.colab import drive
        drive.mount('/content/drive')
    data = load_dataset("json", data_files=json_file_path)["train"]
    return data["prompt"], data

## Inference with Trained DPO Model

In [ ]:
#@title Load New Prompts
DPO_PROMPTS, RAW_PROMPT_DATA = get_dpo_gen_prompts()

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
#@title Load Trained Model
DPO_INFER = AIGobj()

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausa

In [ ]:
#@title Run Inference
RES = DPO_INFER.infer(DPO_PROMPTS, add_system_prompt=True)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
with open('dpo-items--raw--vXX.json', 'w') as f:
    json.dump(RES, f)